# **Data Science and Machine Learning 1 - Year 4 Portfolio**
## Last.fm Data Collection and Preparation
*By Mark Lambert - C00192497 | [LinkedIn](https://www.linkedin.com/in/mark-lambert-27tw721921/) | [GitHub](https://github.com/lambert-27)* | [Personal Email](mailto:marklambert123@gmail.com)

*Lecturer: Ben O'Shaughnessey*

*Submission date: 27 November 2026*

*Start Date: 6th October 2026*

---

## Purpose

This notebook builds the dataset used in the Last.fm listening-history analyses. It collects recent and loved tracks from the Last.fm API, converts the API response into analysis-ready tables, and joins track metadata and audio features from a [Spotify dataset](https://huggingface.co/datasets/GD-Studio/embeat_45m_spotify_tracks) hosted on Hugging Face

## Steps

1. Retrieve recent scrobbles and remove the currently playing entry
2. Clean artist, album, song, and date fields, then local listening-time and daily summary fields
3. Retrieve loved tracks and mark matching scrobbles with `is_loved`
4. Match tracks to Spotify audio features and export the merged dataset

## Files produced

- `datasets/scrobbles.csv` - cleaned scrobbles with loved-track labels
- `datasets/scrobbled_merged_with_hugging_face.csv` - Last.fm data merged with Spotify features for models used in portfolio

The API credentials are read from the local `lastfm_creds.py` file and are not displayed in this notebook

## Portfolio Steps Covered
 This notebooks covers the Data Preparation phase of the CRISP-DM process
 - **Data Understanding**: Data is collected from Last.Fm API and the raw JSON format is examined
 - **Data Preparation**: Cleaning, transforming and compiling data from multiple sources (Last.Fm and Hugging Face for feature extension) to build a final dataset for models

In [3]:
import requests
import pandas as pd
import lastfm_creds

API_KEY = lastfm_creds.API_KEY
USERNAME = lastfm_creds.USERNAME
HF_KEY = lastfm_creds.HF_KEY

In [4]:
all_tracks = []
page = 1
total_pages = 1

while page <= total_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getrecenttracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': page
    })
    data = response.json()
    if page == 1:
        total_pages = int(data['recenttracks']['@attr']['totalPages'])
    all_tracks.extend(data['recenttracks']['track'])
    page += 1

The data's raw columns.

The ones that this portfolio will focus on are **artist, album, name and date**. ```@attr``` holds songs that are marked ```nowplaying = true```. 

As such, anything with this attribute will also be ignored later on.

In [5]:
scrobblesResponse = pd.DataFrame(all_tracks)

scrobblesResponse.columns

Index(['artist', 'streamable', 'image', 'mbid', 'album', 'name', '@attr',
       'url', 'date'],
      dtype='str')

The raw data shows that the actual readable details for each artist, album are embedded with a ```#text``` key in a dictionary, which must be extracted

In [6]:
scrobblesResponse

,artist,streamable,image,mbid,album,name,@attr,url,date
0,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Unfinished Sympathy,{'nowplaying': 'true'},https://www.last.fm/music/Massive+Attack/_/Unf...,NaN
1,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",0094d2a2-d7f1-4dbc-a534-e3d2b441ef48,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Five Man Army - 2012 Mix/Master,NaN,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793640272', '#text': '02 Nov 2026, 1..."
2,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ede2f6c9-ad54-4161-9f00-f815e55ff3d8,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Be Thankful for What You've Got - 2012 Mix/Master,NaN,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639908', '#text': '02 Nov 2026, 1..."
3,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",affc13cc-54c8-4287-ae79-83682a3a9c96,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Blue Lines - 2012 Mix/Master,NaN,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639658', '#text': '02 Nov 2026, 1..."
4,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ea96c118-9de2-4e2a-95c3-f584fb8ebc82,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",One Love - 2012 Mix/Master,NaN,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639396', '#text': '02 Nov 2026, 1..."
...,...,...,...,...,...,...,...,...,...
18738,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",362c85ba-c79b-478e-8321-0383c3ac8101,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,NaN,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18739,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6f93589f-4a47-4a9d-9274-09a0f437ad15,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,NaN,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18740,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6da0b598-d9cf-4f62-8376-bc3181e509af,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,NaN,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18741,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ad79b526-8df8-4e79-8756-0607df153e59,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,NaN,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, as per [Carlos Medina's article](https://medium.com/hemisferios/using-the-last-fm-api-to-extract-data-and-analyzing-it-with-python-part-1-ffe96f812455), first we must select the colums of interest and, as the column name is obscure I have chosen to rename it to **song**

In [7]:
scrobbles = scrobblesResponse

scrobbles = scrobbles[['artist', 'album', 'name', 'url', 'date']]
scrobbles.rename(columns={'name': 'song'}, inplace=True)
scrobbles

,artist,album,song,url,date
0,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Unfinished Sympathy,https://www.last.fm/music/Massive+Attack/_/Unf...,NaN
1,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793640272', '#text': '02 Nov 2026, 1..."
2,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639908', '#text': '02 Nov 2026, 1..."
3,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639658', '#text': '02 Nov 2026, 1..."
4,{'mbid': '10adbe5e-a2c0-4bf3-8249-2b4cbf6e6ca8...,"{'mbid': '', '#text': 'Blue Lines (2012 Mix/Ma...",One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"{'uts': '1793639396', '#text': '02 Nov 2026, 1..."
...,...,...,...,...,...
18738,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18739,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18740,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18741,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, convert the ```artist``` ```album``` and ```date``` to strings

In [8]:
scrobbles['artist'] = scrobbles['artist'].astype(str)
scrobbles['album'] = scrobbles['album'].astype(str)
scrobbles['date'] = scrobbles['date'].astype(str)

Extract the embedded ```#text``` attribute from each of the ```artist, album``` and ```date``` fields

In [9]:
scrobbles[['mbid2','artist']]=scrobbles['artist'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid2', axis=1)
scrobbles[['mbid3','album']]=scrobbles['album'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid3', axis=1)
scrobbles[['uts','date']]=scrobbles['date'].str.split("#text':", n = 1, expand = True)
scrobbles.head()

,artist,album,song,url,date,uts
0,'Massive Attack'},'Blue Lines (2012 Mix/Master)'},Unfinished Sympathy,https://www.last.fm/music/Massive+Attack/_/Unf...,NaN,NaN
1,'Massive Attack'},'Blue Lines (2012 Mix/Master)'},Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:24'}","{'uts': '1793640272', '"
2,'Massive Attack'},'Blue Lines (2012 Mix/Master)'},Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:18'}","{'uts': '1793639908', '"
3,'Massive Attack'},'Blue Lines (2012 Mix/Master)'},Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:14'}","{'uts': '1793639658', '"
4,'Massive Attack'},'Blue Lines (2012 Mix/Master)'},One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:09'}","{'uts': '1793639396', '"


Removing the ```}``` and ```'``` characters from the ```artist and album``` entries

In [10]:
scrobbles['artist'] = scrobbles['artist'].str.strip("}'\" ")
scrobbles['album'] = scrobbles['album'].str.strip("}'\" ")
scrobbles.head()

,artist,album,song,url,date,uts
0,Massive Attack,Blue Lines (2012 Mix/Master),Unfinished Sympathy,https://www.last.fm/music/Massive+Attack/_/Unf...,NaN,NaN
1,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:24'}","{'uts': '1793640272', '"
2,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:18'}","{'uts': '1793639908', '"
3,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:14'}","{'uts': '1793639658', '"
4,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"'02 Nov 2026, 17:09'}","{'uts': '1793639396', '"


The ```date``` and ```uts``` fields also need to be stripped of the remaining dirty characters. I have also chosen to rename uts to **timestamp** as it is more descriptive.

In [11]:
scrobbles['date'] = scrobbles['date'].str.strip("}'\" ")
scrobbles['uts'] = scrobbles['uts'].str.strip("{uts':','\" ")
scrobbles.rename(columns={'uts': 'timestamp'}, inplace=True)
scrobbles.head()

,artist,album,song,url,date,timestamp
0,Massive Attack,Blue Lines (2012 Mix/Master),Unfinished Sympathy,https://www.last.fm/music/Massive+Attack/_/Unf...,NaN,NaN
1,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"02 Nov 2026, 17:24",1793640272
2,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"02 Nov 2026, 17:18",1793639908
3,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"02 Nov 2026, 17:14",1793639658
4,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,"02 Nov 2026, 17:09",1793639396


Next, the date field must be converted into a date only field

In [12]:
scrobbles['date'] = pd.to_datetime(scrobbles['date'], unit='s')
scrobbles['date_only'] = scrobbles['date'].dt.date
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Massive Attack,Blue Lines (2012 Mix/Master),Unfinished Sympathy,https://www.last.fm/music/Massive+Attack/_/Unf...,NaT,NaN,NaT
1,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:24:00,1793640272,2026-11-02
2,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:18:00,1793639908,2026-11-02
3,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:14:00,1793639658,2026-11-02
4,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:09:00,1793639396,2026-11-02
...,...,...,...,...,...,...,...
18738,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18739,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18740,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18741,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Finally, the row which contains NaN must be removed, as this was a song that had the ```@attr``` of ```nowplaying = true```, so it should not count as a fully scrobbled song

In [13]:
scrobbles = scrobbles.dropna(subset=['date'])
scrobbles

,artist,album,song,url,date,timestamp,date_only
1,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:24:00,1793640272,2026-11-02
2,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:18:00,1793639908,2026-11-02
3,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:14:00,1793639658,2026-11-02
4,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:09:00,1793639396,2026-11-02
5,Massive Attack,Blue Lines (2012 Mix/Master),Safe from Harm - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:05:00,1793639108,2026-11-02
...,...,...,...,...,...,...,...
18738,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18739,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18740,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18741,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Then, three new columns- ```hour``` ```tod``` ```month``` are added. First, the time has to be converted to Dublin's time. Then, we can call the time_of_day function to classify between the four periods of a day: **morning, afternoon, evening and night.**

In [14]:
def time_of_day(hour):
    if hour >= 22 or hour < 5:
        return 'night'
    if hour < 12:
        return 'morning'
    if hour < 17:
        return 'afternoon'
    return 'evening'

local_time = scrobbles['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Dublin')
scrobbles['hour'] = local_time.dt.hour
scrobbles['tod'] = scrobbles['hour'].apply(time_of_day)
scrobbles['month'] = scrobbles['date'].dt.to_period('M')
scrobbles.head()

,artist,album,song,url,date,timestamp,date_only,hour,tod,month
1,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:24:00,1793640272,2026-11-02,17,evening,2026-11
2,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:18:00,1793639908,2026-11-02,17,evening,2026-11
3,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:14:00,1793639658,2026-11-02,17,evening,2026-11
4,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:09:00,1793639396,2026-11-02,17,evening,2026-11
5,Massive Attack,Blue Lines (2012 Mix/Master),Safe from Harm - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:05:00,1793639108,2026-11-02,17,evening,2026-11


Using the [unofficial docs](https://lastfm-docs.github.io/api-docs/user/getLovedTracks/#examples) once again, the data can be combined with the ```User.getLovedTracks``` endpoint. A new field called ```is_loved``` can be joined to the full scrobble dataset

In [15]:
all_loved_tracks = []
loved_page = 1
total_loved_pages = 1

while loved_page <= total_loved_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getlovedtracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': loved_page
    })
    data = response.json()
    if loved_page == 1:
        total_loved_pages = int(data['lovedtracks']['@attr']['totalPages'])
    tracks = data['lovedtracks']['track']
    for track in tracks:
        all_loved_tracks.append({
            'artist': track['artist']['name'],
            'track': track['name'],
            'timestamp': int(track['date']['uts']) if 'date' in track else None
        })
    loved_page += 1

Merging the scrobbles and retrieved lovedScrobbles tracks on the columns ```artist``` and ```song``` against ```track``` and adding the new col ```is_loved``` 

In [16]:
lovedScrobbles = pd.DataFrame(all_loved_tracks)
lovedScrobbles['is_loved'] = 1
lovedScrobbles = lovedScrobbles.drop_duplicates(subset=['artist', 'track'])

lovedMerged = pd.merge(
    scrobbles,
    lovedScrobbles[['artist', 'track', 'is_loved']],
    left_on=['artist', 'song'],
    right_on=['artist', 'track'],
    how='left'
)

loved = lovedMerged.drop(columns=['track'])
loved['is_loved'] = loved['is_loved'].fillna(0).astype(int)
loved

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,is_loved
0,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:24:00,1793640272,2026-11-02,17,evening,2026-11,0
1,Massive Attack,Blue Lines (2012 Mix/Master),Be Thankful for What You've Got - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:18:00,1793639908,2026-11-02,17,evening,2026-11,1
2,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:14:00,1793639658,2026-11-02,17,evening,2026-11,0
3,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:09:00,1793639396,2026-11-02,17,evening,2026-11,0
4,Massive Attack,Blue Lines (2012 Mix/Master),Safe from Harm - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:05:00,1793639108,2026-11-02,17,evening,2026-11,1
...,...,...,...,...,...,...,...,...,...,...,...
18644,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,1
18645,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,1
18646,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,1
18647,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,0


In [21]:
loved.to_csv('datasets/scrobbles.csv', index=False)

In order to get more features from the data pulled from Last.Fm, I sourced a large dataset from [HuggingFace](https://huggingface.co/datasets/GD-Studio/embeat_45m_spotify_tracks). 

I had tried previously to use Kaggle spotify datasets, however I was getting a low match rate when merging with my scrobbled data. This dataset has over 45 million entries, and is over 12GB across 3 files. 

As such, the files had to be iterated through in chunks, to bypass malloc errors. 

In [18]:
from data_clean import clean_text, clean_artist

scrobbles['clean_track'] = clean_text(scrobbles['song'])
scrobbles['clean_artist'] = clean_artist(scrobbles['artist'])
scrobbles_unique = scrobbles.drop_duplicates(subset=['clean_artist', 'clean_track']).copy()

I followed this [tutorial](https://huggingface.co/docs/hub/en/datasets-streaming) from huggingface to load the data without manually downloading the files by using `streaming=True`

In [22]:
from datasets import load_dataset
from huggingface_hub import login

login(HF_KEY)
dataset = load_dataset("GD-Studio/embeat_45m_spotify_tracks", split="train", streaming=True)

matched_data = []

for batch in dataset.iter(batch_size=20_000):
    
    data = pd.DataFrame(batch)
    data['clean_track'] = clean_text(data['track_name'])
    data['clean_artist'] = clean_artist(data['artist_name'])
    
    # keep only items in scrobbles_unique
    data = data[
        data['clean_artist'].isin(scrobbles_unique['clean_artist']) & 
        data['clean_track'].isin(scrobbles_unique['clean_track'])
    ]
    
    # if any matched data, add to matched_data
    if len(data) > 0:
        matched_data.append(data)
        
spotify = pd.concat(matched_data, ignore_index=True)
spotify = spotify.drop_duplicates(subset=['clean_artist', 'clean_track'])

final = pd.merge(
    scrobbles_unique,
    spotify.drop(columns=['track_name', 'artist_name', 'album_name']),
    on=['clean_artist', 'clean_track'],
    how='left'
)

The final dataset now has 37 features

In [23]:
final.columns

Index(['artist', 'album', 'song', 'url', 'date', 'timestamp', 'date_only',
       'hour', 'tod', 'month', 'clean_track', 'clean_artist', 'track_id',
       'isrc', 'popularity', 'explicit', 'artist_idx', 'artist_id',
       'artist_popularity', 'artist_genres', 'artist_genre_idx', 'album_id',
       'release_year', 'duration', 'time_signature', 'tempo', 'key', 'mode',
       'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence'],
      dtype='str')

In [24]:
final = final.dropna()

In [25]:
final

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,...,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,Massive Attack,Blue Lines (2012 Mix/Master),Five Man Army - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:24:00,1793640272,2026-11-02,17,evening,2026-11,...,6.0,1.0,0.629,0.6970,-11.293000,0.0765,0.0095,0.2350,0.1180,0.8440
2,Massive Attack,Blue Lines (2012 Mix/Master),Blue Lines - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:14:00,1793639658,2026-11-02,17,evening,2026-11,...,5.0,1.0,0.855,0.4960,-10.572000,0.0824,0.0778,0.4510,0.1200,0.9470
3,Massive Attack,Blue Lines (2012 Mix/Master),One Love - 2012 Mix/Master,https://www.last.fm/music/+noredirect/Massive+...,2026-11-02 17:09:00,1793639396,2026-11-02,17,evening,2026-11,...,9.0,1.0,0.817,0.4330,-10.065000,0.1230,0.0144,0.0009,0.0469,0.6860
15,Portishead,Dummy,Glory Box,https://www.last.fm/music/Portishead/_/Glory+Box,2026-11-02 16:28:00,1793636936,2026-11-02,16,afternoon,2026-11,...,1.0,1.0,0.510,0.4340,-10.004000,0.0361,0.1770,0.5160,0.0603,0.2070
16,Portishead,Dummy,Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,2026-11-02 16:23:00,1793636627,2026-11-02,16,afternoon,2026-11,...,8.0,0.0,0.737,0.4550,-7.997000,0.0805,0.7300,0.0550,0.0717,0.3390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2702,Pink Floyd,The Dark Side of the Moon,Speak to Me,https://www.last.fm/music/Pink+Floyd/_/Speak+t...,2025-11-18 20:59:00,1763499570,2025-11-18,20,evening,2025-11,...,1.0,1.0,0.592,0.0195,-33.349998,0.0369,0.3610,0.8390,0.1080,0.0313
2707,Mike Dean,,INTRO,https://www.last.fm/music/Mike+Dean/_/INTRO,2025-11-18 20:17:00,1763497053,2025-11-18,20,evening,2025-11,...,0.0,1.0,0.368,0.3460,-18.742001,0.0415,0.0478,0.7720,0.1930,0.0376
2708,The Prodigy,,Diesel Power,https://www.last.fm/music/The+Prodigy/_/Diesel...,2025-11-18 16:10:00,1763482248,2025-11-18,16,afternoon,2025-11,...,0.0,1.0,0.725,0.8770,-3.496000,0.0646,0.0680,0.0000,0.0900,0.0976
2709,Eagles,Desperado,Certain Kind of Fool,https://www.last.fm/music/Eagles/_/Certain+Kin...,2025-11-18 15:13:00,1763478822,2025-11-18,15,afternoon,2025-11,...,2.0,1.0,0.659,0.6180,-7.551000,0.0328,0.4080,0.0000,0.3400,0.4670


In [8]:
final.to_csv('datasets/scrobbled_merged_with_hugging_face.csv', index=False)


### Sources
- [HuggingFace spotify-tracks-dataset](https://huggingface.co/datasets/GD-Studio/embeat_45m_spotify_tracks)
- [Last.Fm API](https://www.last.fm/api)